# AdaKV — 加入 H2O / SnapKV 的等预算质量对比

**Phase-2b baseline 扩展**:把 eviction 类 baseline(H2O、SnapKV)加进等预算质量表,
凸显 AdaKV 的 *query-aware-per-decode + 无损 cache* 定位。所有方法走**同一 kernel**、
同预算,唯一差别是选择信号与是否有损。

**Headline 预期**:等预算下 `h2o` / `snapkv` 的 **selection recall 明显低于** `adakv` / `quest`
(needle 若非 prefill heavy-hitter / 不在 observation window,就被永久 evict,decode 再也召不回),
`acc` 同步下掉;`adakv` / `quest` 因每步重选而保住 needle。

**跑之前**:
- Mac 终端先跑 `caffeinate -i`(防睡眠断连,长任务必备)。
- Colab: `Runtime → Change runtime type → T4 GPU`。
- `h2o` 的 prefill 要算累计注意力(O(L²)),比其它方法慢是正常的;`snapkv` 很快。
- 若 runtime 断连,重跑下面的 **Setup** cell 即可(GitHub 是唯一真相源,每次都是 fresh clone)。


## Setup — clone 最新代码 + 安装(每个 session 跑一次)


In [ ]:
%cd /content
!rm -rf adakv && git clone https://github.com/liukefan821/adakv.git
%cd /content/adakv
!pip install -e ".[gpu]" -q
!pip install -U transformers accelerate -q
!git log --oneline -1

In [ ]:
# GPU sanity
import torch
print("CUDA:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv,noheader

## Run A — headline 单针等预算对比(eviction vs re-select)

单针最干净:needle 必须存活才能答对。等预算下看 `recall` 这一列——`h2o`/`snapkv` 应低于 `adakv`/`quest`。


In [ ]:
%cd /content/adakv
!python benchmarks/quality/eval_equal_budget.py \
    --model Qwen/Qwen2.5-1.5B-Instruct \
    --ctx 6000 --needles 6 --trials 4 \
    --budgets 4 6 8 12 \
    --methods full quest adakv h2o snapkv \
    --out results/equal_budget_eviction_6k.csv

### 读表:recall / realized-budget / acc


In [ ]:
import pandas as pd
df = pd.read_csv("results/equal_budget_eviction_6k.csv")
sp = df[df.method != "full"]
print("== selection recall (needle 块被 >=1 head 保住的比例;低方差,先看这个) ==")
print(sp.pivot(index="method", columns="target_budget", values="recall").round(3))
print("\n== realized mean blocks/head (确认真·等预算;h2o/snapkv 会因 recent 窗略高) ==")
print(sp.pivot(index="method", columns="target_budget", values="realized_budget").round(2))
print("\n== answer accuracy ==")
print(sp.pivot(index="method", columns="target_budget", values="accuracy").round(3))
fa = df[df.method=="full"]["accuracy"].values
print(f"\nfull (dense 天花板) acc = {fa[0]:.3f}" if len(fa) else "")

## Run B — 多针(per-head demand > 1)

每个 query 问 3 个 code,单次前向要浮现多个 needle 块,per-head 需求 > 1 且不均——
eviction 在这种更高检索压力下掉得更明显;同时 `adakv_nuc` 的自适应预算旋钮也在这里体现。


In [ ]:
%cd /content/adakv
!python benchmarks/quality/eval_equal_budget.py \
    --model Qwen/Qwen2.5-1.5B-Instruct \
    --ctx 6000 --needles 6 --trials 4 --needles-per-query 3 \
    --budgets 6 8 12 \
    --methods full quest adakv adakv_nuc h2o snapkv \
    --out results/equal_budget_eviction_6k_npq3.csv

In [ ]:
import pandas as pd
df = pd.read_csv("results/equal_budget_eviction_6k_npq3.csv")
sp = df[df.method != "full"]
print("== recall ==");  print(sp.pivot(index="method", columns="target_budget", values="recall").round(3))
print("\n== realized ==");print(sp.pivot(index="method", columns="target_budget", values="realized_budget").round(2))
print("\n== acc ==");    print(sp.pivot(index="method", columns="target_budget", values="accuracy").round(3))

## (可选) 更重的 sweep — 16K 上下文

确认 headline 站得住后再跑。16K 下 `h2o` 的 prefill 累计注意力更慢(O(L²)),耐心等。


In [ ]:
# %cd /content/adakv
# !python benchmarks/quality/eval_equal_budget.py \
#     --model Qwen/Qwen2.5-1.5B-Instruct \
#     --ctx 16000 --needles 8 --trials 6 --needles-per-query 3 \
#     --budgets 6 8 12 \
#     --methods full quest adakv adakv_nuc h2o snapkv \
#     --out results/equal_budget_eviction_16k.csv